# Indoor views of a building

This page reads the indoor elements of one building and draws them in 3D: all levels, one level, a web page, and the walking network alone.

The building is Maunula-talo, a community centre in Helsinki whose rooms, doors and stairwells are mapped on three levels. The repository holds a small extract of it for its tests, `tests/data/maunula.osm.pbf`. With a PBF file of your own, give `OSM` a `bounding_box` around the building you want.

## Read the indoor elements

`read_indoor` takes a `pyrosm.OSM` object and returns an `IndoorElements` object. Its layers are GeoDataFrames in metres, in the UTM zone of the place.

In [1]:
from pyrosm import OSM

import pyrosm3d

osm = OSM("../tests/data/maunula.osm.pbf")
indoor = pyrosm3d.read_indoor(osm)
indoor.areas[["id", "osm_type", "kind", "levels", "tags"]].head()

,id,osm_type,kind,levels,tags
0,1568797319,way,shaft,"(0.0, 1.0, 2.0)","{'access': 'yes', 'highway': 'elevator', 'indo..."
1,1576010525,way,shaft,"(1.0, 2.0)","{'indoor': 'room', 'level': '1;2', 'room': 'st..."
2,1576010526,way,room,"(2.0,)","{'name': 'Luokka 2', 'indoor': 'room', 'level'..."
3,1576010527,way,room,"(2.0,)","{'name': 'Luokka 1', 'indoor': 'room', 'level'..."
4,1576010528,way,room,"(2.0,)","{'name': 'Monitoimiluokka', 'indoor': 'room', ..."


The `areas` are the polygons with an `indoor` tag. Each has a `kind`, which decides how it is drawn, and its `levels`, read from the `level` tag: the lift in the first row has `level=0-2` and is on three levels. `tags` holds an element's OpenStreetMap tags as a dict.

The other layers are `walls` (walls mapped as lines), `doors`, `pois` (shops and amenities that have a level), `buildings` (outlines), and `nodes` and `edges`, the walking network that pyrosm reads with `get_network("walking", nodes=True, indoor_elements=True, walking_areas=True)`.

In [2]:
indoor.areas["kind"].value_counts()

kind
room        42
corridor     8
level        5
shaft        3
wall         1
toilets      1
Name: count, dtype: int64

## Draw all levels

`plot_indoor` returns a plotly figure. Drag it to turn it, and click a legend entry to hide or show a layer.

- Floors are 4 m apart and coloured by the kind of area.
- Walls stand along the outlines of rooms and are cut away at 1.6 m, so that the rooms stay visible from above. A door leaves a gap in the wall.
- Stairwells and lift shafts are columns through the floors that they join.
- The walking network is drawn on its levels, in the colours of `plot_network`.
- Outer walls, doors and shops are hidden at start. A click on their legend entry shows them.

In [3]:
figure = pyrosm3d.plot_indoor(indoor)
figure.show()

## Draw one level

With `level`, the figure holds only what is on that level. The columns of stairwells and lifts are left out, since they pass through several levels.

In [4]:
pyrosm3d.plot_indoor(indoor, level=1).show()

## Write a web page

`to_html` writes the figure as a web page. The page has a button for all levels and one for each level, and a click on a room, a line of the network or a marker shows the element's OpenStreetMap tags beside the figure.

By default the page holds plotly's JavaScript, which makes it about 5 MB larger and usable without a network connection. With `include_plotlyjs="cdn"` the page loads the library from the internet.

In [5]:
pyrosm3d.to_html(indoor, "maunula-talo.html", title="Maunula-talo")

PosixPath('maunula-talo.html')

## Draw the walking network alone

`plot_network` draws a walking network that pyrosm read with `indoor_elements=True`, each edge at its levels. It takes the edges and, for station entrances and routes, the nodes; it also takes a networkx or igraph graph of `get_graph(indoor_elements=True)`. The vertical axis counts levels.

In this extract the streets and platforms are at street level, and the lines across the corridors of Maunula-talo are on levels 1 and 2.

In [6]:
nodes, edges = osm.get_network("walking", nodes=True, indoor_elements=True)
pyrosm3d.plot_network(edges, nodes).show()